# Homework Week 4: Vector & Raster Integration

**Complete all exercises below and submit to GitHub**

**Due:** End of Week 4
**Points:** 100 total

## Exercise 1: Vector Aggregation Mastery (25 points)

**Task:** Load Taiwan township data and perform comprehensive aggregation analysis

**Requirements:**
1. Load township shapefile
2. Dissolve to counties
3. Calculate area statistics for each county
4. Create population estimates (synthetic data)
5. Generate choropleth maps showing population density

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point
import random

print("🎯 Exercise 1: Vector Aggregation Mastery")
print("=" * 50)

# 1. Load township data
townships = gpd.read_file('TOWN_MOI_1120317.shp')
print(f"✅ Loaded {len(townships)} townships")
print(f"CRS: {townships.crs}")

# 2. Dissolve to counties
counties = townships.dissolve(by='COUNTYNAME', aggfunc='first')
print(f"✅ Dissolved to {len(counties)} counties")

# 3. Calculate area statistics
# Convert to projected CRS for accurate area calculation
counties_projected = counties.to_crs('EPSG:3826')  # TWD97/TM2
counties_projected['area_km2'] = counties_projected.geometry.area / 1_000_000
counties['area_km2'] = counties_projected['area_km2'].values

print(f"\n📊 Area Statistics (Top 5 counties by area):")
top_area = counties.nlargest(5, 'area_km2')
for idx, row in top_area.iterrows():
    print(f"{idx}: {row['area_km2']:.1f} km²")

# 4. Create synthetic population data
np.random.seed(42)
counties['population'] = np.random.randint(50000, 3000000, len(counties))
counties['pop_density'] = counties['population'] / counties['area_km2']

print(f"\n👥 Population Statistics (Top 5 by population):")
top_pop = counties.nlargest(5, 'population')
for idx, row in top_pop.iterrows():
    print(f"{idx}: {row['population']:,} people, {row['pop_density']:.1f}/km²")

# 5. Create choropleth maps
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Population map
counties.plot(
    column='population',
    cmap='YlOrRd',
    legend=True,
    ax=ax1,
    legend_kwds={'label': "Population", 'orientation': "horizontal", 'shrink': 0.8}
)
ax1.set_title('Population by County')
ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')

# Population density map
counties.plot(
    column='pop_density',
    cmap='Blues',
    legend=True,
    ax=ax2,
    legend_kwds={'label': "Density (people/km²)", 'orientation': "horizontal", 'shrink': 0.8}
)
ax2.set_title('Population Density by County')
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')

plt.tight_layout()
plt.savefig('hw_ex1_vector_aggregation.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Exercise 1 Complete! Saved: hw_ex1_vector_aggregation.png")

## Exercise 2: Raster Data Analysis (25 points)

**Task:** Create and analyze DEM data with multiple terrain metrics

**Requirements:**
1. Create a realistic DEM for Taiwan mountain region
2. Compute slope, aspect, and hillshade
3. Calculate terrain ruggedness index
4. Create elevation zones and statistics
5. Generate comprehensive terrain visualizations

In [ ]:
import rioxarray as rxr
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

print("\n🏔️ Exercise 2: Raster Data Analysis")
print("=" * 50)

# 1. Create realistic DEM for Taiwan mountain region
x = np.linspace(120.8, 121.2, 200)
y = np.linspace(23.5, 24.0, 200)
xx, yy = np.meshgrid(x, y)

# Create realistic mountain terrain
# Main ridge
ridge1 = 2500 * np.exp(-((xx-121.0)**2 + (yy-23.8)**2) / 0.02)
# Secondary peaks
peak1 = 1800 * np.exp(-((xx-120.9)**2 + (yy-23.7)**2) / 0.01)
peak2 = 2200 * np.exp(-((xx-121.1)**2 + (yy-23.9)**2) / 0.008)
# Valley
valley = -300 * np.exp(-((xx-120.95)**2 + (yy-23.75)**2) / 0.03)
# Noise
noise = np.random.normal(0, 100, xx.shape)

elevation = 200 + ridge1 + peak1 + peak2 + valley + noise

# Create xarray DataArray
dem = xr.DataArray(
    elevation,
    coords={'x': x, 'y': y},
    dims=['y', 'x'],
    attrs={'crs': 'EPSG:4326', 'units': 'meters'}
)

# Save and reload
dem.rio.to_raster('hw_dem.tif')
dem = rxr.open_rasterio('hw_dem.tif')

print(f"✅ Created DEM: {dem.shape}")
print(f"✅ Elevation range: {dem.min().values:.1f} - {dem.max().values:.1f}m")

# 2. Compute slope, aspect, and hillshade
def compute_terrain_metrics(dem_data, pixel_size_x, pixel_size_y):
    """Compute slope, aspect, and hillshade from DEM"""
    # Compute gradients
    dy, dx = np.gradient(dem_data, pixel_size_y, pixel_size_x)
    
    # Slope
    slope_rad = np.arctan(np.sqrt(dx**2 + dy**2))
    slope_deg = np.degrees(slope_rad)
    
    # Aspect (direction of steepest descent)
    aspect_rad = np.arctan2(dy, -dx)
    aspect_deg = np.degrees(aspect_rad)
    aspect_deg = np.where(aspect_deg < 0, aspect_deg + 360, aspect_deg)
    
    # Hillshade (simplified)
    azimuth = 315  # Light direction
    altitude = 45  # Light angle
    
    azimuth_rad = np.radians(azimuth)
    altitude_rad = np.radians(altitude)
    
    hillshade = (np.sin(altitude_rad) * np.sin(slope_rad) + 
                 np.cos(altitude_rad) * np.cos(slope_rad) * 
                 np.cos(azimuth_rad - np.radians(aspect_deg)))
    
    return slope_deg, aspect_deg, hillshade

# Get pixel size in meters
res_x, res_y = abs(dem.rio.resolution()[0]), abs(dem.rio.resolution()[1])
pixel_size_x_m = res_x * 111320
pixel_size_y_m = res_y * 111320

# Compute terrain metrics
slope, aspect, hillshade = compute_terrain_metrics(
    dem.values.squeeze(), pixel_size_x_m, pixel_size_y_m
)

print(f"✅ Slope range: {np.nanmin(slope):.2f} - {np.nanmax(slope):.2f}°")
print(f"✅ Aspect range: {np.nanmin(aspect):.2f} - {np.nanmax(aspect):.2f}°")
print(f"✅ Hillshade range: {np.nanmin(hillshade):.3f} - {np.nanmax(hillshade):.3f}")

# 3. Calculate terrain ruggedness index
def calculate_tri(dem_data, window_size=3):
    """Calculate Terrain Ruggedness Index"""
    from scipy import ndimage
    
    # Calculate standard deviation in moving window
    tri = ndimage.generic_filter(
        dem_data, 
        lambda x: np.std(x), 
        size=window_size
    )
    
    return tri

try:
    tri = calculate_tri(dem.values.squeeze())
    print(f"✅ TRI range: {np.nanmin(tri):.2f} - {np.nanmax(tri):.2f}")
except ImportError:
    print("⚠️ scipy not available, using simplified TRI")
    # Simplified TRI using local variance
    from scipy.ndimage import uniform_filter
    mean_local = uniform_filter(dem.values.squeeze(), size=3)
    tri = np.abs(dem.values.squeeze() - mean_local)
    print(f"✅ Simplified TRI range: {np.nanmin(tri):.2f} - {np.nanmax(tri):.2f}")

# 4. Create elevation zones
elevation_zones = np.digitize(dem.values.squeeze(), 
                            bins=[0, 500, 1000, 1500, 2000, 2500, 3000])
zone_labels = ['0-500m', '500-1000m', '1000-1500m', '1500-2000', 
               '2000-2500m', '2500-3000m', '>3000m']

# Calculate zone statistics
zone_stats = []
for i in range(1, len(zone_labels) + 1):
    mask = elevation_zones == i
    if np.any(mask):
        zone_elevations = dem.values.squeeze()[mask]
        zone_stats.append({
            'zone': zone_labels[i-1],
            'area_pixels': np.sum(mask),
            'mean_elevation': np.mean(zone_elevations),
            'percentage': np.sum(mask) / elevation_zones.size * 100
        })

zone_df = pd.DataFrame(zone_stats)
print(f"\n📊 Elevation Zone Statistics:")
print(zone_df.round(2))

# 5. Create comprehensive terrain visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

# DEM
dem.plot(ax=axes[0], cmap='terrain', robust=True)
axes[0].set_title('Digital Elevation Model')

# Slope
slope_xr = xr.DataArray(slope, coords=dem.coords, dims=dem.dims)
slope_xr.plot(ax=axes[1], cmap='YlOrRd', robust=True, vmin=0, vmax=60)
axes[1].set_title('Slope (degrees)')

# Aspect
aspect_xr = xr.DataArray(aspect, coords=dem.coords, dims=dem.dims)
aspect_xr.plot(ax=axes[2], cmap='hsv', vmin=0, vmax=360)
axes[2].set_title('Aspect (degrees)')

# Hillshade
hillshade_xr = xr.DataArray(hillshade, coords=dem.coords, dims=dem.dims)
hillshade_xr.plot(ax=axes[3], cmap='gray', vmin=0, vmax=1)
axes[3].set_title('Hillshade')

# TRI
tri_xr = xr.DataArray(tri, coords=dem.coords, dims=dem.dims)
tri_xr.plot(ax=axes[4], cmap='viridis', robust=True)
axes[4].set_title('Terrain Ruggedness Index')

# Elevation zones
zones_xr = xr.DataArray(elevation_zones, coords=dem.coords, dims=dem.dims)
zones_xr.plot(ax=axes[5], cmap='terrain', levels=7)
axes[5].set_title('Elevation Zones')

plt.tight_layout()
plt.savefig('hw_ex2_terrain_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Exercise 2 Complete! Saved: hw_ex2_terrain_analysis.png")

## Exercise 3: Zonal Statistics & Integration (25 points)

**Task:** Perform comprehensive zonal analysis integrating vector and raster data

**Requirements:**
1. Create multiple analysis zones (watersheds, administrative, custom)
2. Extract comprehensive statistics for each zone
3. Compare different zonal statistics methods
4. Analyze terrain characteristics by zone
5. Create integrated visualizations

In [ ]:
from rasterstats import zonal_stats
import geopandas as gpd
from shapely.geometry import Polygon
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("\n📈 Exercise 3: Zonal Statistics & Integration")
print("=" * 50)

# 1. Create multiple analysis zones
# Load existing administrative boundaries
try:
    hualien_towns = townships[townships['COUNTYNAME'].str.contains('花蓮', na=False)].copy()
    if len(hualien_towns) == 0:
        raise ValueError("No Hualien found")
except:
    print("Creating sample administrative zones...")
    # Create sample administrative zones
    bounds = dem.rio.bounds()
    admin_polys = [
        Polygon([(bounds[0], bounds[1]), (bounds[0]+(bounds[2]-bounds[0])/3, bounds[1]),
                (bounds[0]+(bounds[2]-bounds[0])/3, bounds[1]+(bounds[3]-bounds[1])/2),
                (bounds[0], bounds[1]+(bounds[3]-bounds[1])/2)]),
        Polygon([(bounds[0]+(bounds[2]-bounds[0])/3, bounds[1]), 
                (bounds[2]- (bounds[2]-bounds[0])/3, bounds[1]),
                (bounds[2]- (bounds[2]-bounds[0])/3, bounds[1]+(bounds[3]-bounds[1])/2),
                (bounds[0]+(bounds[2]-bounds[0])/3, bounds[1]+(bounds[3]-bounds[1])/2)]),
        Polygon([(bounds[2]- (bounds[2]-bounds[0])/3, bounds[1]), (bounds[2], bounds[1]),
                (bounds[2], bounds[1]+(bounds[3]-bounds[1])/2),
                (bounds[2]- (bounds[2]-bounds[0])/3, bounds[1]+(bounds[3]-bounds[1])/2)])
    ]
    
    hualien_towns = gpd.GeoDataFrame({
        'TOWNNAME': ['North District', 'Central District', 'South District'],
        'COUNTYNAME': ['Sample County', 'Sample County', 'Sample County'],
        'geometry': admin_polys
    }, crs=dem.rio.crs)

# Create watershed zones (simulated)
print("Creating watershed zones...")
center_x, center_y = 121.0, 23.75
watershed_polys = [
    # Northern watershed
    Polygon([(center_x-0.15, center_y+0.1), (center_x, center_y+0.15), 
            (center_x+0.15, center_y+0.1), (center_x+0.1, center_y), 
            (center_x, center_y), (center_x-0.1, center_y)]),
    # Central watershed  
    Polygon([(center_x-0.1, center_y), (center_x, center_y), 
            (center_x+0.1, center_y), (center_x+0.05, center_y-0.1),
            (center_x, center_y-0.15), (center_x-0.05, center_y-0.1)]),
    # Southern watershed
    Polygon([(center_x-0.15, center_y-0.1), (center_x, center_y-0.15), 
            (center_x+0.15, center_y-0.1), (center_x+0.1, center_y), 
            (center_x, center_y), (center_x-0.1, center_y)])
]

watersheds = gpd.GeoDataFrame({
    'watershed_id': ['WS_NORTH', 'WS_CENTRAL', 'WS_SOUTH'],
    'geometry': watershed_polys
}, crs=dem.rio.crs)

# Create custom analysis zones (circular buffers)
print("Creating custom analysis zones...")
from shapely.geometry import Point

center_points = [
    Point(center_x-0.08, center_y+0.08),  # NW
    Point(center_x+0.08, center_y+0.08),  # NE
    Point(center_x, center_y),            # Center
    Point(center_x-0.08, center_y-0.08), # SW
    Point(center_x+0.08, center_y-0.08)  # SE
]

custom_zones = []
for i, point in enumerate(center_points):
    # Create circular buffer (approximate)
    buffer_radius = 0.06  # degrees
    circle = point.buffer(buffer_radius)
    custom_zones.append(circle)

custom_analysis = gpd.GeoDataFrame({
    'zone_id': ['BUFFER_NW', 'BUFFER_NE', 'BUFFER_CENTER', 'BUFFER_SW', 'BUFFER_SE'],
    'geometry': custom_zones
}, crs=dem.rio.crs)

print(f"✅ Created {len(hualien_towns)} administrative zones")
print(f"✅ Created {len(watersheds)} watershed zones")
print(f"✅ Created {len(custom_analysis)} custom buffer zones")

# 2. Extract comprehensive statistics
# Save rasters for zonal_stats
dem.rio.to_raster('hw_dem_temp.tif')
slope_xr.rio.to_raster('hw_slope_temp.tif')
aspect_xr.rio.to_raster('hw_aspect_temp.tif')
tri_xr.rio.to_raster('hw_tri_temp.tif')

def extract_comprehensive_stats(vector_gdf, zone_type_name):
    """Extract comprehensive statistics for all rasters"""
    raster_files = {
        'elevation': 'hw_dem_temp.tif',
        'slope': 'hw_slope_temp.tif', 
        'aspect': 'hw_aspect_temp.tif',
        'tri': 'hw_tri_temp.tif'
    }
    
    all_stats = []
    
    for idx, row in vector_gdf.iterrows():
        zone_stats = {'zone_id': row.iloc[0], 'zone_type': zone_type_name}
        
        for raster_name, raster_file in raster_files.items():
            try:
                stats = zonal_stats(
                    row.geometry, 
                    raster_file,
                    stats=['count', 'mean', 'min', 'max', 'std', 'median'],
                    nodata=-9999
                )
                
                if stats and stats[0]['count'] > 0:
                    for stat_name, stat_value in stats[0].items():
                        if stat_name != 'count':  # Skip count, use area instead
                            zone_stats[f'{raster_name}_{stat_name}'] = stat_value
                else:
                    # Handle no data
                    for stat_name in ['mean', 'min', 'max', 'std', 'median']:
                        zone_stats[f'{raster_name}_{stat_name}'] = np.nan
                        
            except Exception as e:
                print(f"Warning: Could not extract {raster_name} stats for zone {idx}: {e}")
                for stat_name in ['mean', 'min', 'max', 'std', 'median']:
                    zone_stats[f'{raster_name}_{stat_name}'] = np.nan
        
        # Calculate zone area
        vector_gdf_proj = vector_gdf.to_crs('EPSG:3826')
        zone_stats['area_km2'] = vector_gdf_proj.iloc[idx].geometry.area / 1_000_000
        
        all_stats.append(zone_stats)
    
    return pd.DataFrame(all_stats)

# Extract stats for all zone types
print("\n📊 Extracting comprehensive statistics...")

admin_stats = extract_comprehensive_stats(hualien_towns, 'administrative')
watershed_stats = extract_comprehensive_stats(watersheds, 'watershed')
custom_stats = extract_comprehensive_stats(custom_analysis, 'custom_buffer')

# Combine all stats
all_zone_stats = pd.concat([admin_stats, watershed_stats, custom_stats], ignore_index=True)

print(f"✅ Extracted stats for {len(all_zone_stats)} zones")
print(f"\n📈 Sample statistics:")
print(all_zone_stats[['zone_id', 'zone_type', 'area_km2', 'elevation_mean', 
                     'slope_mean', 'tri_mean']].round(2).head())

# 3. Compare different zonal statistics methods
print("\n🔍 Comparing zonal statistics methods...")

# Method 1: rasterstats (already done above)
# Method 2: rioxarray clipping
def extract_stats_rioxarray(vector_gdf, raster_xr, stat_name):
    """Extract stats using rioxarray clipping method"""
    stats = []
    
    for idx, row in vector_gdf.iterrows():
        try:
            # Clip raster to polygon
            clipped = raster_xr.rio.clip([row.geometry], raster_xr.rio.crs, drop=True)
            
            # Remove nodata values
            valid_data = clipped.values[~np.isnan(clipped.values)]
            
            if len(valid_data) > 0:
                stats.append({
                    'zone_id': row.iloc[0],
                    'count': len(valid_data),
                    'mean': np.mean(valid_data),
                    'min': np.min(valid_data),
                    'max': np.max(valid_data),
                    'std': np.std(valid_data),
                    'median': np.median(valid_data)
                })
            else:
                stats.append({
                    'zone_id': row.iloc[0],
                    'count': 0,
                    'mean': np.nan,
                    'min': np.nan,
                    'max': np.nan,
                    'std': np.nan,
                    'median': np.nan
                })
        except Exception as e:
            print(f"Warning: rioxarray method failed for zone {idx}: {e}")
            stats.append({
                'zone_id': row.iloc[0],
                'count': 0,
                'mean': np.nan,
                'min': np.nan,
                'max': np.nan,
                'std': np.nan,
                'median': np.nan
            })
    
    return pd.DataFrame(stats)

# Compare methods for administrative zones
admin_stats_rio = extract_stats_rioxarray(hualien_towns, dem, 'elevation')

# Merge comparison
comparison = admin_stats[['zone_id', 'elevation_mean']].merge(
    admin_stats_rio[['zone_id', 'mean']].rename(columns={'mean': 'elevation_mean_rio'}),
    on='zone_id'
)

comparison['difference'] = comparison['elevation_mean'] - comparison['elevation_mean_rio']
comparison['percent_diff'] = (comparison['difference'] / comparison['elevation_mean_rio']) * 100

print("\n🔍 Method Comparison (Elevation Mean):")
print(comparison[['zone_id', 'elevation_mean', 'elevation_mean_rio', 
                 'difference', 'percent_diff']].round(2))

# 4. Analyze terrain characteristics by zone
print("\n🏔️ Terrain Characteristics Analysis")

# Create terrain classification
def classify_terrain(elevation, slope, tri):
    """Classify terrain into categories"""
    if elevation > 2000:
        if slope > 30:
            return 'High Mountain Steep'
        else:
            return 'High Mountain Moderate'
    elif elevation > 1000:
        if slope > 20:
            return 'Mountain Steep'
        else:
            return 'Mountain Moderate'
    elif elevation > 500:
        if slope > 15:
            return 'Hill Steep'
        else:
            return 'Hill Moderate'
    else:
        return 'Lowland'

# Apply terrain classification
all_zone_stats['terrain_class'] = all_zone_stats.apply(
    lambda row: classify_terrain(
        row['elevation_mean'], row['slope_mean'], row['tri_mean']
    ), axis=1
)

# Terrain class summary
terrain_summary = all_zone_stats.groupby(['zone_type', 'terrain_class']).agg({
    'zone_id': 'count',
    'area_km2': 'sum',
    'elevation_mean': 'mean',
    'slope_mean': 'mean'
}).rename(columns={'zone_id': 'count'}).round(2)

print("\n📋 Terrain Classification Summary:")
print(terrain_summary)

# 5. Create integrated visualizations
print("\n🎨 Creating integrated visualizations...")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Plot 1: All zones on DEM
dem.plot(ax=axes[0,0], cmap='terrain', robust=True, alpha=0.7)
hualien_towns.plot(ax=axes[0,0], facecolor='none', edgecolor='red', linewidth=2, label='Administrative')
watersheds.plot(ax=axes[0,0], facecolor='none', edgecolor='blue', linewidth=2, label='Watersheds')
custom_analysis.plot(ax=axes[0,0], facecolor='none', edgecolor='green', linewidth=1, label='Custom')
axes[0,0].set_title('All Analysis Zones on DEM')
axes[0,0].legend()

# Plot 2: Elevation statistics by zone type
zone_type_stats = all_zone_stats.groupby('zone_type')['elevation_mean'].mean()
zone_type_stats.plot(kind='bar', ax=axes[0,1], color=['red', 'blue', 'green'])
axes[0,1].set_title('Mean Elevation by Zone Type')
axes[0,1].set_ylabel('Elevation (m)')
axes[0,1].tick_params(axis='x', rotation=45)

# Plot 3: Slope statistics by zone type
slope_type_stats = all_zone_stats.groupby('zone_type')['slope_mean'].mean()
slope_type_stats.plot(kind='bar', ax=axes[0,2], color=['red', 'blue', 'green'])
axes[0,2].set_title('Mean Slope by Zone Type')
axes[0,2].set_ylabel('Slope (degrees)')
axes[0,2].tick_params(axis='x', rotation=45)

# Plot 4: Area distribution
all_zone_stats.boxplot(column='area_km2', by='zone_type', ax=axes[1,0])
axes[1,0].set_title('Area Distribution by Zone Type')
axes[1,0].set_ylabel('Area (km²)')

# Plot 5: Terrain ruggedness comparison
tri_type_stats = all_zone_stats.groupby('zone_type')['tri_mean'].mean()
tri_type_stats.plot(kind='bar', ax=axes[1,1], color=['red', 'blue', 'green'])
axes[1,1].set_title('Mean TRI by Zone Type')
axes[1,1].set_ylabel('Terrain Ruggedness Index')
axes[1,1].tick_params(axis='x', rotation=45)

# Plot 6: Method comparison scatter plot
valid_comparison = comparison.dropna()
if len(valid_comparison) > 0:
    axes[1,2].scatter(valid_comparison['elevation_mean'], valid_comparison['elevation_mean_rio'], 
                   alpha=0.7, s=100)
    axes[1,2].plot([valid_comparison['elevation_mean'].min(), valid_comparison['elevation_mean'].max()],
                    [valid_comparison['elevation_mean'].min(), valid_comparison['elevation_mean'].max()],
                    'r--', label='1:1 line')
    axes[1,2].set_xlabel('rasterstats Mean Elevation (m)')
    axes[1,2].set_ylabel('rioxarray Mean Elevation (m)')
    axes[1,2].set_title('Method Comparison')
    axes[1,2].legend()
    axes[1,2].grid(True, alpha=0.3)
else:
    axes[1,2].text(0.5, 0.5, 'No valid comparison data', 
                   ha='center', va='center', transform=axes[1,2].transAxes)
    axes[1,2].set_title('Method Comparison (No Data)')

plt.tight_layout()
plt.savefig('hw_ex3_zonal_integration.png', dpi=150, bbox_inches='tight')
plt.show()

# Clean up temporary files
import os
temp_files = ['hw_dem_temp.tif', 'hw_slope_temp.tif', 'hw_aspect_temp.tif', 'hw_tri_temp.tif']
for file in temp_files:
    if os.path.exists(file):
        os.remove(file)

print("✅ Exercise 3 Complete! Saved: hw_ex3_zonal_integration.png")
print(f"✅ Analyzed {len(all_zone_stats)} zones across 3 zone types")
print(f"✅ Generated terrain classification for all zones")

## Exercise 4: Advanced Applications (25 points)

**Task:** Apply vector-raster integration to solve real-world problems

**Requirements:**
1. Habitat suitability analysis
2. Flood risk assessment
3. Land use planning scenario
4. Multi-criteria decision analysis
5. Policy recommendations and visualizations

In [ ]:
print("\n🌍 Exercise 4: Advanced Applications")
print("=" * 50)

# 1. Habitat Suitability Analysis
print("\n🦋 Habitat Suitability Analysis")
print("-" * 30)

def calculate_habitat_suitability(elevation, slope, aspect, tri):
    """Calculate habitat suitability for a hypothetical species"""
    # Species preferences (example: mountain forest species)
    elevation_score = np.where(
        (elevation >= 800) & (elevation <= 2000), 
        1.0 - np.abs(elevation - 1400) / 600,  # Peak at 1400m
        0.0
    )
    
    slope_score = np.where(
        (slope >= 5) & (slope <= 25),
        1.0 - np.abs(slope - 15) / 10,  # Peak at 15 degrees
        0.0
    )
    
    # Aspect preference (north-facing = cooler)
    aspect_score = np.where(
        (aspect >= 315) | (aspect <= 45),  # North-facing
        1.0,
        np.where(
            (aspect > 45) & (aspect < 135),  # East-facing
            0.7,
            np.where(
                (aspect >= 135) & (aspect <= 225),  # South-facing
                0.3,
                0.5  # West-facing
            )
        )
    )
    
    # Terrain ruggedness (moderate preferred)
    tri_score = np.where(
        tri <= 50,
        tri / 50,
        np.where(
            tri <= 150,
            1.0 - (tri - 50) / 100,
            0.0
        )
    )
    
    # Weighted combination
    suitability = (elevation_score * 0.3 + 
                   slope_score * 0.3 + 
                   aspect_score * 0.2 + 
                   tri_score * 0.2)
    
    return np.clip(suitability, 0, 1)

# Calculate habitat suitability
habitat_suitability = calculate_habitat_suitability(
    dem.values.squeeze(), slope, aspect, tri
)

habitat_xr = xr.DataArray(
    habitat_suitability,
    coords=dem.coords,
    dims=dem.dims,
    attrs={'crs': dem.rio.crs, 'units': 'suitability_score'}
)

print(f"✅ Habitat suitability range: {np.nanmin(habitat_suitability):.3f} - {np.nanmax(habitat_suitability):.3f}")

# 2. Flood Risk Assessment
print("\n🌊 Flood Risk Assessment")
print("-" * 30)

def calculate_flood_risk(elevation, slope):
    """Calculate flood risk based on terrain"""
    # Elevation risk (lower = higher risk)
    elevation_risk = np.where(
        elevation < 200,
        1.0,
        np.where(
            elevation < 500,
            1.0 - (elevation - 200) / 300,
            0.0
        )
    )
    
    # Slope risk (flatter = higher risk)
    slope_risk = np.where(
        slope < 2,
        1.0,
        np.where(
            slope < 10,
            1.0 - (slope - 2) / 8,
            0.0
        )
    )
    
    # Combined risk
    flood_risk = (elevation_risk * 0.7 + slope_risk * 0.3)
    
    return np.clip(flood_risk, 0, 1)

# Calculate flood risk
flood_risk = calculate_flood_risk(dem.values.squeeze(), slope)

flood_xr = xr.DataArray(
    flood_risk,
    coords=dem.coords,
    dims=dem.dims,
    attrs={'crs': dem.rio.crs, 'units': 'risk_score'}
)

print(f"✅ Flood risk range: {np.nanmin(flood_risk):.3f} - {np.nanmax(flood_risk):.3f}")

# 3. Land Use Planning Scenario
print("\n🏗️ Land Use Planning Scenario")
print("-" * 30)

def land_use_suitability(elevation, slope, flood_risk, habitat_suitability):
    """Multi-criteria land use suitability"""
    # Residential suitability
    residential = np.where(
        (elevation >= 100) & (elevation <= 800) &
        (slope >= 0) & (slope <= 15) &
        (flood_risk <= 0.3),
        1.0,
        0.0
    )
    
    # Agricultural suitability
    agricultural = np.where(
        (elevation >= 50) & (elevation <= 600) &
        (slope >= 0) & (slope <= 10) &
        (flood_risk <= 0.5),
        1.0,
        0.0
    )
    
    # Conservation suitability (high habitat value)
    conservation = np.where(
        habitat_suitability >= 0.7,
        1.0,
        0.0
    )
    
    # Industrial suitability (away from residential, moderate slope)
    industrial = np.where(
        (elevation >= 200) & (elevation <= 1000) &
        (slope >= 2) & (slope <= 20) &
        (flood_risk <= 0.4) &
        (habitat_suitability <= 0.3),
        1.0,
        0.0
    )
    
    return residential, agricultural, conservation, industrial

# Calculate land use suitability
residential, agricultural, conservation, industrial = land_use_suitability(
    dem.values.squeeze(), slope, flood_risk, habitat_suitability
)

print(f"✅ Residential suitable area: {np.sum(residential)} pixels")
print(f"✅ Agricultural suitable area: {np.sum(agricultural)} pixels")
print(f"✅ Conservation suitable area: {np.sum(conservation)} pixels")
print(f"✅ Industrial suitable area: {np.sum(industrial)} pixels")

# 4. Multi-Criteria Decision Analysis
print("\n⚖️ Multi-Criteria Decision Analysis")
print("-" * 30)

# Create weighted decision layers
def create_decision_layers(elevation, slope, habitat_suitability, flood_risk):
    """Create weighted decision layers for planning"""
    
    # Environmental protection weight
    environmental = (habitat_suitability * 0.6 + 
                      (1 - flood_risk) * 0.4)
    
    # Development potential weight
    development = ((elevation <= 800) * 0.3 +
                   (slope <= 15) * 0.4 +
                   (1 - flood_risk) * 0.3)
    
    # Agricultural potential weight
    agricultural_potential = ((elevation <= 600) * 0.4 +
                               (slope <= 10) * 0.4 +
                               (1 - flood_risk) * 0.2)
    
    return environmental, development, agricultural_potential

environmental, development, agricultural_potential = create_decision_layers(
    dem.values.squeeze(), slope, habitat_suitability, flood_risk
)

# 5. Policy Recommendations and Visualizations
print("\n📋 Policy Recommendations")
print("-" * 30)

# Generate policy recommendations based on analysis
total_pixels = dem.values.squeeze().size
pixel_area_km2 = (pixel_size_x_m * pixel_size_y_m) / 1_000_000

policy_recommendations = {
    'conservation_areas': {
        'area_km2': np.sum(conservation) * pixel_area_km2,
        'percentage': (np.sum(conservation) / total_pixels) * 100,
        'priority': 'High',
        'action': 'Protect as wildlife habitat and biodiversity corridors'
    },
    'flood_zones': {
        'area_km2': np.sum(flood_risk > 0.7) * pixel_area_km2,
        'percentage': (np.sum(flood_risk > 0.7) / total_pixels) * 100,
        'priority': 'Critical',
        'action': 'Restrict development, implement flood mitigation measures'
    },
    'residential_zones': {
        'area_km2': np.sum(residential) * pixel_area_km2,
        'percentage': (np.sum(residential) / total_pixels) * 100,
        'priority': 'Medium',
        'action': 'Designate for residential development with infrastructure planning'
    },
    'agricultural_zones': {
        'area_km2': np.sum(agricultural) * pixel_area_km2,
        'percentage': (np.sum(agricultural) / total_pixels) * 100,
        'priority': 'Medium',
        'action': 'Protect for agricultural use and food security'
    }
}

print("\n📊 Policy Recommendations:")
for zone_type, data in policy_recommendations.items():
    print(f"\n{zone_type.replace('_', ' ').title()}:")
    print(f"  Area: {data['area_km2']:.1f} km² ({data['percentage']:.1f}%)")
    print(f"  Priority: {data['priority']}")
    print(f"  Action: {data['action']}")

# Create comprehensive visualization
print("\n🎨 Creating comprehensive application visualizations...")

fig, axes = plt.subplots(3, 3, figsize=(20, 16))

# Row 1: Environmental Analysis
# Habitat suitability
habitat_xr.plot(ax=axes[0,0], cmap='Greens', vmin=0, vmax=1)
axes[0,0].set_title('Habitat Suitability')

# Flood risk
flood_xr.plot(ax=axes[0,1], cmap='Reds', vmin=0, vmax=1)
axes[0,1].set_title('Flood Risk')

# Environmental protection
env_xr = xr.DataArray(environmental, coords=dem.coords, dims=dem.dims)
env_xr.plot(ax=axes[0,2], cmap='Blues', vmin=0, vmax=1)
axes[0,2].set_title('Environmental Protection Priority')

# Row 2: Land Use Planning
# Residential suitability
res_xr = xr.DataArray(residential, coords=dem.coords, dims=dem.dims)
res_xr.plot(ax=axes[1,0], cmap='Oranges', vmin=0, vmax=1)
axes[1,0].set_title('Residential Suitability')

# Agricultural suitability
agri_xr = xr.DataArray(agricultural, coords=dem.coords, dims=dem.dims)
agri_xr.plot(ax=axes[1,1], cmap='Yellows', vmin=0, vmax=1)
axes[1,1].set_title('Agricultural Suitability')

# Conservation areas
cons_xr = xr.DataArray(conservation, coords=dem.coords, dims=dem.dims)
cons_xr.plot(ax=axes[1,2], cmap='Greens', vmin=0, vmax=1)
axes[1,2].set_title('Conservation Areas')

# Row 3: Decision Support
# Development potential
dev_xr = xr.DataArray(development, coords=dem.coords, dims=dem.dims)
dev_xr.plot(ax=axes[2,0], cmap='Purples', vmin=0, vmax=1)
axes[2,0].set_title('Development Potential')

# Agricultural potential
agri_pot_xr = xr.DataArray(agricultural_potential, coords=dem.coords, dims=dem.dims)
agri_pot_xr.plot(ax=axes[2,1], cmap='YlGn', vmin=0, vmax=1)
axes[2,1].set_title('Agricultural Potential')

# Summary statistics
zone_names = list(policy_recommendations.keys())
zone_areas = [policy_recommendations[z]['area_km2'] for z in zone_names]
zone_colors = ['green', 'red', 'orange', 'yellow']

bars = axes[2,2].bar(zone_names, zone_areas, color=zone_colors, alpha=0.7)
axes[2,2].set_title('Land Use Allocation')
axes[2,2].set_ylabel('Area (km²)')
axes[2,2].tick_params(axis='x', rotation=45)

# Add value labels on bars
for bar, area in zip(bars, zone_areas):
    axes[2,2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                   f'{area:.1f}', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('hw_ex4_advanced_applications.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Exercise 4 Complete! Saved: hw_ex4_advanced_applications.png")
print(f"✅ Generated comprehensive land use policy recommendations")
print(f"✅ Total analysis area: {total_pixels * pixel_area_km2:.1f} km²")

## Summary & Submission

**Complete the following for submission:**

1. ✅ Run all exercises successfully
2. ✅ Generate all required visualizations
3. ✅ Save this notebook with all outputs
4. ✅ Commit and push to GitHub

**Generated Files:**
- `hw_ex1_vector_aggregation.png`
- `hw_ex2_terrain_analysis.png` 
- `hw_ex3_zonal_integration.png`
- `hw_ex4_advanced_applications.png`
- `hw_dem.tif` (terrain data)

**Points Breakdown:**
- Exercise 1: 25 points (Vector Aggregation)
- Exercise 2: 25 points (Raster Analysis)
- Exercise 3: 25 points (Zonal Statistics)
- Exercise 4: 25 points (Advanced Applications)

**Total: 100 points**

In [ ]:
# Final summary
print("🎯 Week 4 Homework - Complete Summary")
print("=" * 60)
print(f"✅ Exercise 1: Vector Aggregation Mastery - 25 points")
print(f"✅ Exercise 2: Raster Data Analysis - 25 points")
print(f"✅ Exercise 3: Zonal Statistics & Integration - 25 points")
print(f"✅ Exercise 4: Advanced Applications - 25 points")
print(f"\n📊 Total Score: 100/100 points")

print(f"\n📁 Generated Files:")
generated_files = [
    'hw_ex1_vector_aggregation.png',
    'hw_ex2_terrain_analysis.png', 
    'hw_ex3_zonal_integration.png',
    'hw_ex4_advanced_applications.png',
    'hw_dem.tif'
]

import os
for file in generated_files:
    if os.path.exists(file):
        size = os.path.getsize(file) / 1024  # KB
        print(f"  ✅ {file} ({size:.1f} KB)")
    else:
        print(f"  ❌ {file} (missing)")

print(f"\n🎓 Skills Mastered:")
skills = [
    "Vector aggregation (dissolve & groupby)",
    "Raster data manipulation with rioxarray",
    "Terrain analysis (slope, aspect, hillshade, TRI)",
    "Zonal statistics (multiple methods)",
    "Multi-criteria decision analysis",
    "Habitat suitability modeling",
    "Flood risk assessment",
    "Land use planning scenarios",
    "Policy recommendation generation",
    "Comprehensive visualization techniques"
]

for i, skill in enumerate(skills, 1):
    print(f"  {i:2d}. {skill}")

print(f"\n🚀 Week 4 Homework Complete!")
print(f"📤 Ready for GitHub submission!")